# Laboratorio 14 — Qué cambió desde ayer

**Duración: 45 minutos.**

Un proceso nocturno consume tu tabla todas las noches y la copia a otro sistema. Anoche
había treinta mil filas y hoy hay treinta mil una. ¿Tiene que volver a leer las treinta
mil para encontrar la que cambió?

La respuesta de hoy es que no, y lo que lo hace posible ya lo tienes: **la libreta sabe
qué trajo cada página**.

---

### En el almacén

Hasta ahora usaste la libreta para mirar **cómo estaba la tabla** en una página: eso es el
`VERSION AS OF` del laboratorio 01, y te devuelve la foto completa.

Hoy vas a pedirle otra cosa, y es distinta: **qué anotaciones se hicieron entre una página
y otra**. No la foto, sino el movimiento. Las filas que entraron, las que salieron y las
que cambiaron, cada una con su etiqueta.


## Paso 0 — La tabla de hoy

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero.

In [ ]:
%%sql
-- Celda 0.2



Las tres columnas de siempre, y un padrón chico para poder seguirlo con la vista.

In [ ]:
%%sql
-- Celda 0.3



Los cuatro contribuyentes. **Esta es la carga de anoche**, y la página que deja es el
punto de partida de todo lo que viene.

In [ ]:
%%sql
-- Celda 0.4



## Paso 1 — Un día de movimientos

Ahora pasa el día y la tabla recibe tres cosas distintas, cada una por su lado. Primero
llegan dos contribuyentes nuevos.

In [ ]:
%%sql
-- Celda 1.1



Después, a uno lo reclasifican.

In [ ]:
%%sql
-- Celda 1.2



Y por último, a otro lo dan de baja del padrón.

In [ ]:
%%sql
-- Celda 1.3



Tres movimientos, tres páginas nuevas. Con la carga inicial, cuatro.

**Fíjate bien en la lista**: la segunda página es el cierre de anoche, la última es donde
estamos ahora. Esos dos números son los que vas a usar en el paso 3.

In [ ]:
%%sql
-- Celda 1.4



## Paso 2 — La vista de cambios

Ahora le pedimos a Iceberg el movimiento, no la foto. El procedimiento
`create_changelog_view` construye una **vista** con una fila por cada cambio.

Dos avisos antes de ejecutar, porque los dos sorprenden:

- La vista **vive en tu sesión**, no en el catálogo. No la vas a encontrar en
  `SHOW TABLES`, y si reinicias el kernel desaparece. Se llama
  `contribuyentes_lab14_changes`, con el nombre de la tabla y el sufijo.
- **Un `UPDATE` no aparece como una modificación.** Aparece como un `DELETE` de la fila
  vieja más un `INSERT` de la nueva, en la misma página. Así es como Iceberg guarda una
  corrección, y así te la devuelve.

In [ ]:
%%sql
-- Celda 2.1



Ahora consulta la vista. Toda la historia de la tabla, movimiento a movimiento.

Las tres columnas nuevas son las que importan:

- `_change_type` dice qué pasó: `INSERT` o `DELETE`.
- `_change_ordinal` numera las páginas desde cero, para poder ordenar.
- `_commit_snapshot_id` dice en qué página ocurrió, y enlaza con `.snapshots`.

In [ ]:
%%sql
-- Celda 2.2



## Paso 3 — Solo lo de hoy

Ahí está todo, desde el principio de los tiempos. Pero un proceso nocturno no quiere
todo: quiere **lo que pasó desde la última vez que corrió**.

Para eso el mismo procedimiento acepta un rango. `start-snapshot-id` es la página desde la
cual mirar, **sin incluirla**, y `end-snapshot-id` es hasta dónde.

Pon en el primero la **segunda** página de tu lista, que es el cierre de anoche, y en el
segundo la **última**. Y dale otro nombre a la vista para no pisar la anterior.

In [ ]:
%%sql
-- Celda 3.1



Y ahora consulta esa.

In [ ]:
%%sql
-- Celda 3.2



## Paso 4 — Cómo lo usa un proceso nocturno

**Tres filas**, y son exactamente el reclasificado y la baja. Los dos contribuyentes
nuevos no están, porque entraron en la página que pusiste como punto de partida y esa no
se incluye.

Esto es la **lectura incremental**, y así la usa un proceso de verdad:

1. La primera vez lee la tabla entera y **guarda el `snapshot_id` que procesó**. En un
   archivo, en una tabla de control, donde sea, pero lo guarda.
2. La noche siguiente pregunta cuál es la página actual, pide el changelog **entre la que
   guardó y esa**, y aplica solo eso en el destino.
3. Si todo salió bien, guarda el nuevo `snapshot_id` y se duerme.

Tres propiedades que hacen que esto funcione en producción, y conviene nombrarlas:

- **No relee nada.** El costo de la carga nocturna depende de lo que cambió, no del
  tamaño de la tabla. Es la diferencia entre minutos y horas.
- **Es repetible.** Si el proceso se cae a mitad, vuelve a pedir el mismo rango y obtiene
  lo mismo. No hay que adivinar por dónde iba.
- **No hace falta tocar la tabla de origen.** No hay columna de auditoría que mantener, ni
  disparadores, ni una tabla de bitácora al lado. La información ya estaba en la libreta.

Y una advertencia, que es la que muerde: **el rango deja de existir si alguien expira esas
páginas**. Si el laboratorio 11 te dejó con ganas de `expire_snapshots`, ten presente que
la retención tiene que ser más larga que el intervalo del proceso que consume. Si el
proceso corre cada noche y expiras a diario, un fin de semana largo te deja sin rango.

## Cierre

Dos formas de preguntarle a la libreta, y no se parecen:

| Lo que pides | Lo que te devuelve | Para qué sirve |
|---|---|---|
| `VERSION AS OF` | la **foto** completa en una página | auditar, comparar, revertir |
| `create_changelog_view` | el **movimiento** entre dos páginas | alimentar a otro sistema |

La primera la viste en el laboratorio 01 y responde *cómo estaba*. La segunda responde
*qué pasó*, y es la que convierte una tabla Iceberg en una **fuente** para el resto de la
plataforma.

Ese es el punto que cierra el laboratorio. La DGT no tiene una sola herramienta: tiene
tableros, procesos de carga y sistemas que consumen. Con esto, ninguno necesita una copia
propia de la tabla ni un proceso que la recorra entera cada noche. Piden lo que cambió, y
lo que cambió estaba escrito desde el principio.
